In [1]:

import sys
import os

print("Python version:", sys.version)
print("Lokasi kerja:", os.getcwd())

print("\nGoogle Colab berhasil dijalankan!")


Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Lokasi kerja: /content

Google Colab berhasil dijalankan!


In [2]:

!pip install -q streamlit scikit-learn pandas numpy joblib matplotlib seaborn


In [3]:

import os

folders = [
    "dataset",
    "models",
    "notebook"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Struktur folder berhasil dibuat!")

for folder in folders:
    print("📁", folder)


Struktur folder berhasil dibuat!
📁 dataset
📁 models
📁 notebook


In [4]:

import os
import shutil

# Repository resmi mata kuliah
repo_url = "https://github.com/Muhammad-Ikhwan-Fathulloh/Artificial-Intelegent-Course-Bank.git"

# Mengunduh repository
!git clone --depth 1 {repo_url} /content/course_bank

# Lokasi dataset
source = "/content/course_bank/UTS/dataset/kasus09_mode_choice.csv"
destination = "/content/dataset/kasus09_mode_choice.csv"

# Memindahkan dataset ke folder proyek
if os.path.exists(source):
    shutil.copy2(source, destination)
    print("Dataset berhasil diunduh dan disimpan!")
else:
    print("Dataset tidak ditemukan. Periksa struktur repository.")


Cloning into '/content/course_bank'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 38 (delta 5), reused 27 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 417.93 KiB | 4.22 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Dataset berhasil diunduh dan disimpan!


In [5]:

import pandas as pd
import os

# Membaca dataset
file_path = "/content/dataset/kasus09_mode_choice.csv"

df = pd.read_csv(file_path)

# Menampilkan informasi dataset
print("NAMA DATASET: kasus09_mode_choice.csv")
print("=" * 45)

print("Ukuran dataset:", df.shape)

print("\n5 Data Pertama:")
display(df.head())

print("\nInformasi Dataset:")
df.info()

print("\nNama Kolom:")
print(df.columns.tolist())


NAMA DATASET: kasus09_mode_choice.csv
Ukuran dataset: (840, 9)

5 Data Pertama:


,individual,mode,choice,ttme,invc,invt,gc,hinc,psize
0,1.0,1.0,0.0,69.0,59.0,100.0,70.0,35.0,1.0
1,1.0,2.0,0.0,34.0,31.0,372.0,71.0,35.0,1.0
2,1.0,3.0,0.0,35.0,25.0,417.0,70.0,35.0,1.0
3,1.0,4.0,1.0,0.0,10.0,180.0,30.0,35.0,1.0
4,2.0,1.0,0.0,64.0,58.0,68.0,68.0,30.0,2.0



Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   individual  840 non-null    float64
 1   mode        840 non-null    float64
 2   choice      840 non-null    float64
 3   ttme        840 non-null    float64
 4   invc        840 non-null    float64
 5   invt        840 non-null    float64
 6   gc          840 non-null    float64
 7   hinc        840 non-null    float64
 8   psize       840 non-null    float64
dtypes: float64(9)
memory usage: 59.2 KB

Nama Kolom:
['individual', 'mode', 'choice', 'ttme', 'invc', 'invt', 'gc', 'hinc', 'psize']


In [6]:

print("=== AUDIT DATASET ===")

# Jumlah data duplikat
print("\nJumlah data duplikat:")
print(df.duplicated().sum())

# Missing value
print("\nMissing value:")
print(df.isnull().sum())

# Jumlah individu
print("\nJumlah individu:")
print(df["individual"].nunique())

# Distribusi kode moda
print("\nDistribusi kode moda:")
print(df["mode"].value_counts().sort_index())

# Distribusi pilihan
print("\nDistribusi choice:")
print(df["choice"].value_counts().sort_index())


=== AUDIT DATASET ===

Jumlah data duplikat:
0

Missing value:
individual    0
mode          0
choice        0
ttme          0
invc          0
invt          0
gc            0
hinc          0
psize         0
dtype: int64

Jumlah individu:
210

Distribusi kode moda:
mode
1.0    210
2.0    210
3.0    210
4.0    210
Name: count, dtype: int64

Distribusi choice:
choice
0.0    630
1.0    210
Name: count, dtype: int64


In [7]:

%%writefile fitur.py

import pandas as pd


def feature_engineering(df):

    # =====================================
    # 1. VALIDASI STRUKTUR DATA
    # =====================================

    required_columns = [
        'individual', 'mode', 'choice',
        'ttme', 'invc', 'invt', 'gc',
        'hinc', 'psize'
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Kolom tidak ditemukan: {missing_columns}"
        )

    data = df[required_columns].copy()

    if data.isnull().any().any():
        raise ValueError("Dataset memiliki missing value.")

    # =====================================
    # 2. VALIDASI DATA INDIVIDU
    # =====================================

    # Setiap individu harus memiliki 4 alternatif
    jumlah_mode = data.groupby(
        'individual'
    )['mode'].nunique()

    if not (jumlah_mode == 4).all():
        raise ValueError(
            "Ada individu yang tidak memiliki 4 alternatif moda."
        )

    # Memastikan tidak ada kombinasi individu dan moda duplikat
    if data.duplicated(
        subset=['individual', 'mode']
    ).any():
        raise ValueError(
            "Terdapat kombinasi individu dan moda duplikat."
        )

    # Memastikan setiap individu memiliki tepat satu pilihan
    jumlah_pilihan = data.groupby(
        'individual'
    )['choice'].sum()

    if not (jumlah_pilihan == 1).all():
        raise ValueError(
            "Setiap individu harus memiliki tepat satu choice = 1."
        )

    # =====================================
    # 3. MEMERIKSA KONSISTENSI DATA
    # =====================================

    for col in ['hinc', 'psize']:
        konsistensi = data.groupby(
            'individual'
        )[col].nunique()

        if not (konsistensi == 1).all():
            raise ValueError(
                f"Nilai {col} tidak konsisten."
            )

    # =====================================
    # 4. MEMBUAT TARGET
    # =====================================

    target = (
        data[data['choice'] == 1]
        [['individual', 'mode']]
        .rename(columns={'mode': 'target'})
    )

    # =====================================
    # 5. MENGUBAH LONG KE WIDE FORMAT
    # =====================================

    # Fitur individu
    individu = data.groupby(
        'individual'
    ).agg({
        'hinc': 'first',
        'psize': 'first'
    })

    # Fitur masing-masing moda
    atribut = ['ttme', 'invc', 'invt', 'gc']

    wide_data = []

    for atribut_name in atribut:

        pivot = data.pivot(
            index='individual',
            columns='mode',
            values=atribut_name
        )

        pivot.columns = [
            f'{atribut_name}_mode_{int(mode)}'
            for mode in pivot.columns
        ]

        wide_data.append(pivot)

    # Menggabungkan seluruh fitur
    X = pd.concat(
        [individu] + wide_data,
        axis=1
    )

    # =====================================
    # 6. MENGGABUNGKAN TARGET
    # =====================================

    hasil = X.join(
        target.set_index('individual')
    )

    hasil = hasil.reset_index()

    # =====================================
    # 7. VALIDASI HASIL
    # =====================================

    if hasil.isnull().any().any():
        raise ValueError(
            "Hasil transformasi memiliki missing value."
        )

    if hasil['individual'].duplicated().any():
        raise ValueError(
            "Terdapat individu duplikat setelah transformasi."
        )

    return hasil


Writing fitur.py


In [8]:

import pandas as pd

from fitur import feature_engineering

# Membaca dataset awal
df = pd.read_csv(
    'dataset/kasus09_mode_choice.csv'
)

# Menjalankan Feature Engineering
data_wide = feature_engineering(df)

# Menampilkan hasil
print("Transformasi berhasil!")
print("Ukuran data awal:", df.shape)
print("Ukuran data setelah transformasi:", data_wide.shape)

print("\nLima data pertama:")
display(data_wide.head())

print("\nDaftar kolom:")
print(data_wide.columns.tolist())


Transformasi berhasil!
Ukuran data awal: (840, 9)
Ukuran data setelah transformasi: (210, 20)

Lima data pertama:


,individual,hinc,psize,ttme_mode_1,ttme_mode_2,ttme_mode_3,ttme_mode_4,invc_mode_1,invc_mode_2,invc_mode_3,invc_mode_4,invt_mode_1,invt_mode_2,invt_mode_3,invt_mode_4,gc_mode_1,gc_mode_2,gc_mode_3,gc_mode_4,target
0,1.0,35.0,1.0,69.0,34.0,35.0,0.0,59.0,31.0,25.0,10.0,100.0,372.0,417.0,180.0,70.0,71.0,70.0,30.0,4.0
1,2.0,30.0,2.0,64.0,44.0,53.0,0.0,58.0,31.0,25.0,11.0,68.0,354.0,399.0,255.0,68.0,84.0,85.0,50.0,4.0
2,3.0,40.0,1.0,69.0,34.0,35.0,0.0,115.0,98.0,53.0,23.0,125.0,892.0,882.0,720.0,129.0,195.0,149.0,101.0,4.0
3,4.0,70.0,3.0,64.0,44.0,53.0,0.0,49.0,26.0,21.0,5.0,68.0,354.0,399.0,180.0,59.0,79.0,81.0,32.0,4.0
4,5.0,45.0,2.0,64.0,44.0,53.0,0.0,60.0,32.0,26.0,8.0,144.0,404.0,449.0,600.0,82.0,93.0,94.0,99.0,4.0



Daftar kolom:
['individual', 'hinc', 'psize', 'ttme_mode_1', 'ttme_mode_2', 'ttme_mode_3', 'ttme_mode_4', 'invc_mode_1', 'invc_mode_2', 'invc_mode_3', 'invc_mode_4', 'invt_mode_1', 'invt_mode_2', 'invt_mode_3', 'invt_mode_4', 'gc_mode_1', 'gc_mode_2', 'gc_mode_3', 'gc_mode_4', 'target']


In [9]:

print("=== VALIDASI HASIL FEATURE ENGINEERING ===")

print("\nJumlah individu:")
print(data_wide['individual'].nunique())

print("\nMissing value:")
print(data_wide.isnull().sum().sum())

print("\nDuplikasi individu:")
print(data_wide['individual'].duplicated().sum())

print("\nDistribusi target:")
print(data_wide['target'].value_counts().sort_index())

print("\nJumlah fitur:")
print(len(data_wide.columns) - 2)

print("\nJumlah alternatif per individu:")
print(4)


=== VALIDASI HASIL FEATURE ENGINEERING ===

Jumlah individu:
210

Missing value:
0

Duplikasi individu:
0

Distribusi target:
target
1.0    58
2.0    63
3.0    30
4.0    59
Name: count, dtype: int64

Jumlah fitur:
18

Jumlah alternatif per individu:
4


In [10]:

%%writefile latih.py

import os
import json
import joblib
import numpy as np
import pandas as pd

from fitur import feature_engineering

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt


# ==========================================
# 1. KONFIGURASI
# ==========================================

RANDOM_STATE = 42

DATA_PATH = "dataset/kasus09_mode_choice.csv"

MODEL_DIR = "models"

os.makedirs(MODEL_DIR, exist_ok=True)


# ==========================================
# 2. MEMBACA DATASET DAN FEATURE ENGINEERING
# ==========================================

print("=" * 55)
print("TAHAP 1: FEATURE ENGINEERING")
print("=" * 55)

df = pd.read_csv(DATA_PATH)

data = feature_engineering(df)

print("Ukuran dataset awal:", df.shape)
print("Ukuran setelah transformasi:", data.shape)


# ==========================================
# 3. MENENTUKAN FITUR DAN TARGET
# ==========================================

X = data.drop(
    columns=["individual", "target"]
)

y = data["target"].astype(int)

feature_names = X.columns.tolist()

print("\nJumlah fitur:", len(feature_names))
print("Jumlah kelas:", y.nunique())

print("\nDistribusi target:")
print(y.value_counts().sort_index())


# ==========================================
# 4. MEMBAGI DATA TRAINING DAN TESTING
# ==========================================

print("\n" + "=" * 55)
print("TAHAP 2: TRAIN-TEST SPLIT")
print("=" * 55)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Data training:", X_train.shape)
print("Data testing:", X_test.shape)

print("\nDistribusi target training:")
print(y_train.value_counts().sort_index())

print("\nDistribusi target testing:")
print(y_test.value_counts().sort_index())


# ==========================================
# 5. MEMBUAT PIPELINE MODEL
# ==========================================

print("\n" + "=" * 55)
print("TAHAP 3: PEMBUATAN MODEL")
print("=" * 55)

model_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

model_rf = Pipeline([
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE
    ))
])

models = {
    "Logistic Regression": model_lr,
    "Random Forest": model_rf
}

print("Logistic Regression berhasil dibuat.")
print("Random Forest berhasil dibuat.")


# ==========================================
# 6. CROSS VALIDATION
# ==========================================

print("\n" + "=" * 55)
print("TAHAP 4: STRATIFIED 5-FOLD CROSS VALIDATION")
print("=" * 55)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

hasil_cv = []

for nama, model in models.items():

    print(f"\nEvaluasi: {nama}")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    hasil_cv.append({
        "Model": nama,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "F1 Macro Mean": scores["test_f1_macro"].mean(),
        "F1 Macro Std": scores["test_f1_macro"].std()
    })


hasil_cv_df = pd.DataFrame(hasil_cv)

print("\nHasil Cross Validation:")
print(
    hasil_cv_df.sort_values(
        by="F1 Macro Mean",
        ascending=False
    ).to_string(index=False)
)


# ==========================================
# 7. MEMILIH MODEL TERBAIK
# ==========================================

print("\n" + "=" * 55)
print("TAHAP 5: PEMILIHAN MODEL")
print("=" * 55)

best_name = hasil_cv_df.sort_values(
    by="F1 Macro Mean",
    ascending=False
).iloc[0]["Model"]

best_model = models[best_name]

print("Model terbaik:", best_name)


# ==========================================
# 8. TRAINING MODEL TERBAIK
# ==========================================

print("\n" + "=" * 55)
print("TAHAP 6: FINAL TRAINING")
print("=" * 55)

best_model.fit(X_train, y_train)

print("Training berhasil.")
print("Jumlah data training:", len(X_train))


# ==========================================
# 9. EVALUASI DATA TESTING SATU KALI
# ==========================================

print("\n" + "=" * 55)
print("TAHAP 7: EVALUASI AKHIR DATA TESTING")
print("=" * 55)

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("\nHASIL EVALUASI")
print("-" * 35)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

print("\nClassification Report:")

report = classification_report(
    y_test,
    y_pred,
    zero_division=0
)

print(report)


# ==========================================
# 10. CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=best_model.classes_
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=best_model.classes_
)

disp.plot(cmap="Blues")

plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.show()


# ==========================================
# 11. MENYIMPAN MODEL DAN METADATA
# ==========================================

print("\n" + "=" * 55)
print("TAHAP 8: DEPLOYMENT JOBLIB")
print("=" * 55)

metadata = {
    "model_name": best_name,
    "dataset": "kasus09_mode_choice.csv",
    "target_name": "target",
    "feature_names": feature_names,
    "class_labels": [
        int(x) for x in best_model.classes_
    ],
    "metrics": {
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1)
    }
}

artifact = {
    "pipeline": best_model,
    "metadata": metadata
}

model_path = os.path.join(
    MODEL_DIR,
    "random_forest_mode_choice.joblib"
)

joblib.dump(artifact, model_path)

# Menyimpan hasil evaluasi
hasil_cv_df.to_csv(
    os.path.join(MODEL_DIR, "hasil_cross_validation.csv"),
    index=False
)

with open(
    os.path.join(MODEL_DIR, "metadata.json"),
    "w"
) as file:
    json.dump(metadata, file, indent=4)

print("Model berhasil disimpan!")
print("Lokasi:", model_path)
print("Jumlah fitur:", len(feature_names))
print("Model terpilih:", best_name)

print("\nSEMUA PROSES SELESAI!")


Writing latih.py


In [11]:

!python latih.py


TAHAP 1: FEATURE ENGINEERING
Ukuran dataset awal: (840, 9)
Ukuran setelah transformasi: (210, 20)

Jumlah fitur: 18
Jumlah kelas: 4

Distribusi target:
target
1    58
2    63
3    30
4    59
Name: count, dtype: int64

TAHAP 2: TRAIN-TEST SPLIT
Data training: (168, 18)
Data testing: (42, 18)

Distribusi target training:
target
1    47
2    50
3    24
4    47
Name: count, dtype: int64

Distribusi target testing:
target
1    11
2    13
3     6
4    12
Name: count, dtype: int64

TAHAP 3: PEMBUATAN MODEL
Logistic Regression berhasil dibuat.
Random Forest berhasil dibuat.

TAHAP 4: STRATIFIED 5-FOLD CROSS VALIDATION

Evaluasi: Logistic Regression

Evaluasi: Random Forest

Hasil Cross Validation:
              Model  Accuracy Mean  Accuracy Std  F1 Macro Mean  F1 Macro Std
      Random Forest       0.946881      0.043131       0.931520      0.076844
Logistic Regression       0.750624      0.078679       0.745531      0.094241

TAHAP 5: PEMILIHAN MODEL
Model terbaik: Random Forest

TAHAP 6: FI

In [12]:

import os

print("=== PEMERIKSAAN FILE MODEL ===")

files = [
    "fitur.py",
    "latih.py",
    "models/random_forest_mode_choice.joblib",
    "models/metadata.json",
    "models/hasil_cross_validation.csv"
]

for file in files:
    status = os.path.exists(file)
    print(
        ("[OK] " if status else "[TIDAK DITEMUKAN] ") + file
    )


=== PEMERIKSAAN FILE MODEL ===
[OK] fitur.py
[OK] latih.py
[OK] models/random_forest_mode_choice.joblib
[OK] models/metadata.json
[OK] models/hasil_cross_validation.csv


In [13]:

%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import os

# =====================================
# KONFIGURASI
# =====================================

st.set_page_config(
    page_title="Mode Choice Prediction",
    page_icon="🚆",
    layout="wide"
)

MODEL_PATH = "models/random_forest_mode_choice.joblib"

# =====================================
# MEMUAT MODEL
# =====================================

@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            "File model tidak ditemukan. Jalankan latih.py terlebih dahulu."
        )

    return joblib.load(MODEL_PATH)

try:
    artifact = load_model()

    model = artifact["pipeline"]
    metadata = artifact["metadata"]

    features = metadata["feature_names"]
    metrics = metadata["metrics"]
    classes = metadata["class_labels"]

except Exception as e:
    st.error(f"Gagal memuat model: {e}")
    st.stop()


# =====================================
# HEADER
# =====================================

st.title("🚆 Mode Choice Prediction")

st.write(
    "Sistem klasifikasi pilihan moda transportasi "
    "menggunakan algoritma Random Forest."
)

st.caption(
    "Dataset: kasus09_mode_choice.csv"
)

st.divider()


# =====================================
# DASHBOARD EVALUASI
# =====================================

st.subheader("Dashboard Evaluasi Model")

st.write(f"Model yang digunakan: **{metadata['model_name']}**")

col1, col2, col3, col4 = st.columns(4)

col1.write("Accuracy")
col1.subheader(
    f"{metrics['accuracy'] * 100:.2f}%"
)

col2.write("Precision")
col2.subheader(
    f"{metrics['precision_macro'] * 100:.2f}%"
)

col3.write("Recall")
col3.subheader(
    f"{metrics['recall_macro'] * 100:.2f}%"
)

col4.write("F1-Score")
col4.subheader(
    f"{metrics['f1_macro'] * 100:.2f}%"
)

st.caption(
    "Metrik merupakan hasil evaluasi akhir menggunakan data testing."
)

st.divider()


# =====================================
# FORM INPUT PREDIKSI
# =====================================

st.subheader("Prediksi Pilihan Moda Transportasi")

st.write(
    "Masukkan karakteristik individu dan atribut "
    "dari masing-masing alternatif transportasi."
)

input_data = {}

with st.form("form_prediksi"):

    st.markdown("### A. Karakteristik Individu")

    col1, col2 = st.columns(2)

    with col1:
        input_data["hinc"] = st.number_input(
            "Pendapatan Rumah Tangga (hinc)",
            value=35.0
        )

    with col2:
        input_data["psize"] = st.number_input(
            "Ukuran Rombongan (psize)",
            min_value=1,
            value=1
        )

    st.divider()

    st.markdown("### B. Atribut Alternatif Transportasi")

    for mode in range(1, 5):

        st.markdown(f"#### Alternatif Moda {mode}")

        col1, col2 = st.columns(2)

        with col1:
            input_data[f"ttme_mode_{mode}"] = st.number_input(
                f"Waktu Tempuh Terminal - Moda {mode}",
                value=0.0,
                key=f"ttme_{mode}"
            )

            input_data[f"invt_mode_{mode}"] = st.number_input(
                f"Waktu Dalam Kendaraan - Moda {mode}",
                value=0.0,
                key=f"invt_{mode}"
            )

        with col2:
            input_data[f"invc_mode_{mode}"] = st.number_input(
                f"Biaya Dalam Kendaraan - Moda {mode}",
                value=0.0,
                key=f"invc_{mode}"
            )

            input_data[f"gc_mode_{mode}"] = st.number_input(
                f"Biaya Gabungan - Moda {mode}",
                value=0.0,
                key=f"gc_{mode}"
            )

    st.divider()

    submitted = st.form_submit_button(
        "Prediksi Pilihan Transportasi",
        use_container_width=True
    )


# =====================================
# HASIL PREDIKSI
# =====================================

if submitted:

    # Menyesuaikan urutan fitur dengan model
    input_df = pd.DataFrame(
        [[input_data[col] for col in features]],
        columns=features
    )

    prediction = model.predict(input_df)[0]

    probabilities = model.predict_proba(input_df)[0]

    st.divider()

    st.subheader("Hasil Prediksi")

    st.success(
        f"Kode moda transportasi yang diprediksi: {int(prediction)}"
    )

    # Tabel probabilitas
    probability_df = pd.DataFrame({
        "Kode Moda": classes,
        "Probabilitas": probabilities
    })

    probability_df["Persentase"] = (
        probability_df["Probabilitas"] * 100
    )

    st.markdown("### Probabilitas Setiap Alternatif")

    st.dataframe(
        probability_df[
            ["Kode Moda", "Persentase"]
        ].style.format({
            "Persentase": "{:.2f}%"
        }),
        use_container_width=True,
        hide_index=True
    )

    # Grafik probabilitas
    st.markdown("### Grafik Probabilitas")

    chart_df = probability_df.set_index("Kode Moda")

    st.bar_chart(chart_df["Probabilitas"])

    st.info(
        "Prediksi menunjukkan kelas dengan probabilitas "
        "tertinggi berdasarkan model Random Forest."
    )


# =====================================
# FOOTER
# =====================================

st.divider()

st.caption(
    "Mode Choice Prediction | Artificial Intelligence | "
    "Random Forest Classification"
)


Writing app.py


In [14]:

import os

print("app.py tersedia:", os.path.exists("app.py"))
print("Ukuran file:", os.path.getsize("app.py"), "bytes")


app.py tersedia: True
Ukuran file: 5317 bytes


In [15]:
import os

files = [
    "fitur.py",
    "latih.py",
    "app.py",
    "models/random_forest_mode_choice.joblib",
    "models/metadata.json",
    "models/hasil_cross_validation.csv"
]

for file in files:
    if os.path.exists(file):
        print(f"[OK] {file}")
    else:
        print(f"[BELUM ADA] {file}")

[OK] fitur.py
[OK] latih.py
[OK] app.py
[OK] models/random_forest_mode_choice.joblib
[OK] models/metadata.json
[OK] models/hasil_cross_validation.csv


In [26]:
import subprocess
import time
import requests

# Jalankan Streamlit di background
log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Tunggu server mulai berjalan
time.sleep(5)

# Periksa status server
try:
    response = requests.get("http://127.0.0.1:8501/_stcore/health")
    print("Status HTTP:", response.status_code)
    print("Respons:", response.text)
except Exception as e:
    print("Server belum berhasil dijalankan:", e)

Status HTTP: 200
Respons: ok


In [17]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.3
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.3
npm notice To update run: npm install -g npm@12.2.3
npm notice
⠇

In [27]:
!lt --port 8501

your url is: https://cyan-dingos-hear.loca.lt
^C


In [19]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Cloudflare Tunnel berhasil disiapkan.")

Cloudflare Tunnel berhasil disiapkan.


In [28]:
import requests
import subprocess

# Periksa apakah Streamlit merespons
try:
    r = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=5
    )
    print("HTTP Status:", r.status_code)
    print("Respons:", r.text)
except Exception as e:
    print("Streamlit tidak merespons:", e)

# Periksa proses Streamlit
result = subprocess.run(
    ["pgrep", "-af", "streamlit"],
    capture_output=True,
    text=True
)
print("\nProses Streamlit:")
print(result.stdout if result.stdout else "Tidak ditemukan")

Streamlit tidak merespons: HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: /_stcore/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7cc306a2d810>: Failed to establish a new connection: [Errno 111] Connection refused'))

Proses Streamlit:
7452 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true



In [29]:
with open("/content/streamlit.log", "r") as f:
    print(f.read()[-10000:])



2026-11-25 08:10:03.263 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.71.7.81:8501

  Stopping...



In [31]:
!./cloudflared tunnel --url http://127.0.0.1:8501

2026-11-25T08:13:09Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-25T08:13:09Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-25T08:13:12Z INF +--------------------------------------------------------------------------------------------+
2026-11-25T08:13:12Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-25T08:13:12Z INF |  https://better-counties-writing-intro.trycloudflare.c

In [32]:
with open("app.py", "r") as file:
    print(file.read())


import streamlit as st
import pandas as pd
import joblib
import os

# =====================================
# KONFIGURASI
# =====================================

st.set_page_config(
    page_title="Mode Choice Prediction",
    page_icon="🚆",
    layout="wide"
)

MODEL_PATH = "models/random_forest_mode_choice.joblib"

# =====================================
# MEMUAT MODEL
# =====================================

@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            "File model tidak ditemukan. Jalankan latih.py terlebih dahulu."
        )

    return joblib.load(MODEL_PATH)

try:
    artifact = load_model()

    model = artifact["pipeline"]
    metadata = artifact["metadata"]

    features = metadata["feature_names"]
    metrics = metadata["metrics"]
    classes = metadata["class_labels"]

except Exception as e:
    st.error(f"Gagal memuat model: {e}")
    st.stop()


# =====================================
# HEADER
# 

In [33]:

%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os

# =====================================
# KONFIGURASI
# =====================================

st.set_page_config(
    page_title="Mode Choice Prediction",
    page_icon="🚆",
    layout="wide"
)

MODEL_PATH = "models/random_forest_mode_choice.joblib"

# =====================================
# MEMUAT MODEL
# =====================================

@st.cache_resource
def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            "File model tidak ditemukan. Jalankan latih.py terlebih dahulu."
        )

    return joblib.load(MODEL_PATH)


try:
    artifact = load_model()

    model = artifact["pipeline"]
    metadata = artifact["metadata"]

    features = metadata["feature_names"]
    metrics = metadata["metrics"]
    classes = metadata["class_labels"]

except Exception as e:
    st.error(f"Gagal memuat model: {e}")
    st.stop()


# =====================================
# HEADER
# =====================================

st.title("🚆 Mode Choice Prediction")

st.write(
    "Sistem klasifikasi pilihan moda transportasi "
    "menggunakan algoritma Random Forest."
)

st.caption("Dataset: kasus09_mode_choice.csv")

st.divider()


# =====================================
# DASHBOARD EVALUASI
# =====================================

st.subheader("Dashboard Evaluasi Model")

st.write(f"Model yang digunakan: **{metadata['model_name']}**")

col1, col2, col3, col4 = st.columns(4)

col1.write("Accuracy")
col1.subheader(f"{metrics['accuracy'] * 100:.2f}%")

col2.write("Precision")
col2.subheader(f"{metrics['precision_macro'] * 100:.2f}%")

col3.write("Recall")
col3.subheader(f"{metrics['recall_macro'] * 100:.2f}%")

col4.write("F1-Score")
col4.subheader(f"{metrics['f1_macro'] * 100:.2f}%")

st.caption(
    "Metrik merupakan hasil evaluasi akhir menggunakan data testing."
)

st.divider()


# =====================================
# FORM INPUT PREDIKSI
# =====================================

st.subheader("Prediksi Pilihan Moda Transportasi")

st.write(
    "Masukkan karakteristik individu dan atribut dari "
    "keempat alternatif transportasi yang tersedia."
)

st.info(
    "Satu pengisian mewakili satu individu dengan empat alternatif "
    "transportasi. Masukkan nilai sesuai data yang ingin diprediksi."
)

input_data = {}

with st.form("form_prediksi"):

    # ---------------------------------
    # A. KARAKTERISTIK INDIVIDU
    # ---------------------------------

    st.markdown("### A. Karakteristik Individu")

    col1, col2 = st.columns(2)

    with col1:
        input_data["hinc"] = st.number_input(
            "Pendapatan Rumah Tangga (hinc)",
            value=35.0,
            step=0.1,
            format="%.2f",
            help="Nilai pendapatan rumah tangga sesuai format dataset."
        )

    with col2:
        input_data["psize"] = st.number_input(
            "Ukuran Rombongan (psize)",
            min_value=1,
            value=1,
            step=1,
            format="%d",
            help="Jumlah anggota dalam rombongan."
        )

    st.divider()

    # ---------------------------------
    # B. ATRIBUT ALTERNATIF TRANSPORTASI
    # ---------------------------------

    st.markdown("### B. Atribut Alternatif Transportasi")

    st.caption(
        "Setiap alternatif memiliki empat atribut: waktu tempuh terminal, "
        "biaya dalam kendaraan, waktu dalam kendaraan, dan biaya gabungan."
    )

    for mode in range(1, 5):

        st.markdown(f"#### Alternatif Transportasi {mode}")

        col1, col2 = st.columns(2)

        with col1:
            input_data[f"ttme_mode_{mode}"] = st.number_input(
                f"Waktu Tempuh Terminal (ttme) - Alternatif {mode}",
                value=0.0,
                step=0.1,
                format="%.2f",
                key=f"ttme_{mode}",
                help="Waktu tempuh menuju terminal."
            )

            input_data[f"invt_mode_{mode}"] = st.number_input(
                f"Waktu Dalam Kendaraan (invt) - Alternatif {mode}",
                value=0.0,
                step=0.1,
                format="%.2f",
                key=f"invt_{mode}",
                help="Waktu yang dihabiskan di dalam kendaraan."
            )

        with col2:
            input_data[f"invc_mode_{mode}"] = st.number_input(
                f"Biaya Dalam Kendaraan (invc) - Alternatif {mode}",
                value=0.0,
                step=0.1,
                format="%.2f",
                key=f"invc_{mode}",
                help="Biaya yang dikeluarkan dalam kendaraan."
            )

            input_data[f"gc_mode_{mode}"] = st.number_input(
                f"Biaya Gabungan (gc) - Alternatif {mode}",
                value=0.0,
                step=0.1,
                format="%.2f",
                key=f"gc_{mode}",
                help="Generalized cost atau biaya gabungan."
            )

    st.divider()

    submitted = st.form_submit_button(
        "Prediksi Pilihan Transportasi",
        use_container_width=True
    )


# =====================================
# HASIL PREDIKSI
# =====================================

if submitted:

    # Memastikan urutan fitur sesuai model
    input_df = pd.DataFrame(
        [[input_data[col] for col in features]],
        columns=features
    )

    prediction = model.predict(input_df)[0]
    probabilities = model.predict_proba(input_df)[0]

    st.divider()
    st.subheader("Hasil Prediksi")

    st.success(
        f"Kode moda transportasi yang diprediksi: {int(prediction)}"
    )

    # Tabel probabilitas
    probability_df = pd.DataFrame({
        "Kode Moda": classes,
        "Probabilitas": probabilities
    })

    probability_df["Persentase"] = (
        probability_df["Probabilitas"] * 100
    )

    st.markdown("### Probabilitas Setiap Alternatif")

    st.dataframe(
        probability_df[["Kode Moda", "Persentase"]].style.format({
            "Persentase": "{:.2f}%"
        }),
        use_container_width=True,
        hide_index=True
    )

    # Grafik probabilitas
    st.markdown("### Grafik Probabilitas")

    chart_df = probability_df.set_index("Kode Moda")

    st.bar_chart(chart_df["Probabilitas"])

    st.info(
        "Prediksi menunjukkan kelas dengan probabilitas tertinggi "
        "berdasarkan model Random Forest."
    )


# =====================================
# FOOTER
# =====================================

st.divider()

st.caption(
    "Mode Choice Prediction | Artificial Intelligence | "
    "Random Forest Classification"
)


Overwriting app.py


In [34]:
import os
print("Ukuran app.py:", os.path.getsize("app.py"), "bytes")

Ukuran app.py: 6581 bytes


In [35]:
import requests

try:
    r = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=5
    )

    print("Status:", r.status_code)
    print("Respons:", r.text)

except Exception as e:
    print("Server belum aktif:", e)

Status: 200
Respons: ok


In [36]:
import subprocess
import time
import requests
import os

# Hentikan proses Streamlit lama
subprocess.run("pkill -f 'streamlit run app.py'", shell=True)

time.sleep(2)

# Jalankan Streamlit di background
subprocess.Popen(
    "nohup streamlit run app.py "
    "--server.port 8501 "
    "--server.address 0.0.0.0 "
    "--server.headless true "
    "> /content/streamlit.log 2>&1 &",
    shell=True,
    start_new_session=True
)

# Tunggu server aktif
for i in range(15):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )
        if response.status_code == 200:
            print("BERHASIL: Streamlit aktif!")
            print("Status:", response.status_code)
            print("Respons:", response.text)
            break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    print("Streamlit belum aktif. Periksa log:")
    print(open("/content/streamlit.log").read()[-5000:])

BERHASIL: Streamlit aktif!
Status: 200
Respons: ok


In [37]:
import subprocess
import requests
import os

# Periksa server Streamlit
try:
    r = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=5
    )
    print("Streamlit:", r.status_code, r.text)
except Exception as e:
    print("Streamlit error:", e)

# Periksa proses Cloudflare
result = subprocess.run(
    ["pgrep", "-af", "cloudflared"],
    capture_output=True,
    text=True
)

print("\nProses Cloudflare:")
print(result.stdout if result.stdout else "Cloudflare tidak berjalan")

Streamlit: 200 ok

Proses Cloudflare:
Cloudflare tidak berjalan


In [38]:
!./cloudflared tunnel --url http://127.0.0.1:8501

2026-11-25T08:31:27Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-25T08:31:27Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-25T08:31:31Z INF +--------------------------------------------------------------------------------------------+
2026-11-25T08:31:31Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-25T08:31:31Z INF |  https://detailed-lesser-somewhat-missouri.trycloudfla

In [1]:
import shutil

shutil.make_archive(
    "/content/Case09_Mode_Choice",
    "zip",
    "/content",
    "Case09_Mode_Choice"
)

print("ZIP berhasil dibuat!")


FileNotFoundError: [Errno 2] No such file or directory: '/content/Case09_Mode_Choice'

In [2]:
import os

print("Daftar isi /content:")
print(os.listdir("/content"))

Daftar isi /content:
['.config', 'cloudflared', 'dataset', 'streamlit.log', '__pycache__', 'latih.py', 'fitur.py', 'app.py', 'Case09_Mode_Choice.zip', 'course_bank', 'models', 'notebook', 'sample_data']


In [3]:
import os

folders = [
    "/content/dataset",
    "/content/models",
    "/content/notebook"
]

for folder in folders:
    print(f"\nISI FOLDER: {folder}")
    if os.path.exists(folder):
        for file in os.listdir(folder):
            print("-", file)
    else:
        print("Folder tidak ditemukan")

print("\nFILE UTAMA:")
for file in ["fitur.py", "latih.py", "app.py"]:
    print(file, "Tersedia" if os.path.exists("/content/" + file) else "Tidak ditemukan")


ISI FOLDER: /content/dataset
- kasus09_mode_choice.csv

ISI FOLDER: /content/models
- metadata.json
- hasil_cross_validation.csv
- random_forest_mode_choice.joblib

ISI FOLDER: /content/notebook

FILE UTAMA:
fitur.py Tersedia
latih.py Tersedia
app.py Tersedia
